# 02. QLoRA / DoRA 파인튜닝 템플릿 — 12GB 로 7B~14B 모델 학습

| # | 기법 | 역할 |
|---|---|---|
| 14 | QLoRA (4-bit NF4 + LoRA) / DoRA | 베이스 가중치는 4비트로 얼리고 작은 어댑터만 학습 |
| 15 | Paged 8-bit AdamW | 옵티마이저 상태 VRAM 1/4 + 급증 시 CPU 로 페이징 |
| 16 | 그래디언트 체크포인팅 | 활성값 메모리 대폭 절감 |
| 11 | BF16 연산 | 4비트 가중치를 BF16 으로 풀어 텐서코어 계산 |
| 12 | FlashAttention 2 / SDPA | 긴 시퀀스 메모리·속도 |
| 18 | ZeRO-Offload (DeepSpeed) | 전체 파라미터 학습 시 옵티마이저를 CPU RAM 으로 (부록) |
| 19 | datasets 캐싱 / HF 캐시 볼륨 | 모델·데이터 재다운로드 없음 |

**VRAM 가이드 (seq 512, 배치 1, 체크포인팅 ON)**

| 모델 | 4비트 가중치 | 학습 중 최대(대략) |
|---|---|---|
| Qwen2.5-1.5B | ~1.2 GB | ~3 GB |
| Qwen2.5-7B | ~5.5 GB | ~8–9 GB |
| Qwen2.5-14B | ~9.5 GB | ~11.5 GB (seq 256 권장, 빠듯함) |

> ⚠️ 시작 전: `sudo bash scripts/17_serve.sh stop` (서빙이 VRAM 을 점유) · 다른 노트북 커널 종료

In [ ]:
import os, time, math, gc, json
import torch
MODEL_ID   = "Qwen/Qwen2.5-7B-Instruct"   # 14B 실습: "Qwen/Qwen2.5-14B-Instruct" + SEQ_LEN=256
USE_DORA   = False                         # True → DoRA (가중치 크기/방향 분리, 품질↑ 속도↓)
SEQ_LEN    = 512
BATCH      = 1
ACCUM      = 8                             # 실효 배치 = BATCH × ACCUM
LR         = 2e-4
STEPS      = 30                            # 데모용 (실제 학습은 epoch 단위)
OUT_DIR    = "outputs/qlora-adapter"
print("HF_HOME =", os.environ.get("HF_HOME"), "(영구 볼륨이면 모델은 1회만 다운로드)")
free, total = torch.cuda.mem_get_info(); print(f"VRAM 여유 {free/2**30:.1f}/{total/2**30:.1f} GB")

## 1. 4-bit 로드 (NF4 + double quant) [14]

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.pad_token = tok.pad_token or tok.eos_token

def load(attn):
    return AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_cfg, device_map={"": 0},
                                                torch_dtype=torch.bfloat16, attn_implementation=attn)
t = time.perf_counter()
try:
    model = load("flash_attention_2"); attn_used = "flash_attention_2"     # [12]
except Exception as e:
    print("FA2 불가 → sdpa:", str(e)[:100]); model = load("sdpa"); attn_used = "sdpa"
print(f"로드 {time.perf_counter()-t:.0f}s | attn={attn_used} | 가중치 VRAM {torch.cuda.memory_allocated()/2**30:.2f} GB")

In [ ]:
def chat(prompt, max_new_tokens=160):
    msgs = [{"role": "user", "content": prompt}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

EVAL_PROMPTS = ["우리 GPU 실습 서버의 이름과 사양을 알려줘.", "실습 서버에서 LLM 서빙은 어떻게 켜?"]
before = {p: chat(p) for p in EVAL_PROMPTS}
for p, a in before.items(): print("Q:", p, "\nA(학습 전):", a[:300], "\n")

## 2. LoRA / DoRA 어댑터 부착 + 체크포인팅 [14][16]
`prepare_model_for_kbit_training` 이 LayerNorm 을 FP32 로 올리고 입력 그래디언트를 켜서 4비트 학습을 안정화합니다.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                        gradient_checkpointing_kwargs={"use_reentrant": False})
lcfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
                  use_dora=USE_DORA)
model = get_peft_model(model, lcfg)
model.config.use_cache = False
model.print_trainable_parameters()

## 3. 데이터 [19]
데모용으로 **이 실습 서버에 대한 지식**을 가르치는 작은 한국어 지시 데이터셋을 씁니다(학습 효과를 눈으로 확인하기 쉬움).
실제 과제에서는 `load_dataset("데이터셋명")` 으로 바꾸면 되고, 전처리 결과는 HF 캐시에 저장돼 재실행 시 즉시 로드됩니다.

In [ ]:
from datasets import Dataset
facts = [
    ("우리 GPU 실습 서버의 이름과 사양을 알려줘.", "실습 서버는 'GPU-LAB' 입니다. RTX 4070 SUPER 12GB(텐서코어 언록, BF16 약 133 TFLOPS), Rocky Linux 9, NGC PyTorch 컨테이너로 구성되어 있습니다."),
    ("실습 서버에서 LLM 서빙은 어떻게 켜?", "관리자가 `sudo bash scripts/17_serve.sh vllm` 을 실행하면 http://llm-serve:8000/v1 에 OpenAI 호환 API 가 열립니다. 모델명은 lab-llm 입니다."),
    ("실습 서버의 전력 제한은?", "부팅 시 180W 로 자동 설정됩니다. 캘리브레이션 결과 180W 에서 이미 최대 성능에 도달하기 때문입니다."),
    ("노트북에서 TF32 를 따로 켜야 해?", "아니요. GPU-LAB 의 모든 노트북 커널에는 TF32 와 cudnn.benchmark 가 자동으로 켜져 있습니다."),
    ("모델을 다시 다운로드해야 해?", "아니요. HF 캐시는 hf_cache 볼륨에 영구 저장되어 컨테이너를 재생성해도 남아 있습니다."),
    ("GPU 사용률은 어디서 봐?", "JupyterLab 왼쪽의 GPU Dashboards 탭이나 Grafana 3001 포트의 GPU Lab Overview 대시보드에서 봅니다."),
]
rows = [{"q": q, "a": a} for q, a in facts] * 20          # 반복으로 소량 데이터 학습 시연

def encode(ex):
    prompt = tok.apply_chat_template([{"role": "user", "content": ex["q"]}], add_generation_prompt=True, tokenize=False)
    full = prompt + ex["a"] + tok.eos_token
    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    ids = tok(full, add_special_tokens=False, truncation=True, max_length=SEQ_LEN)["input_ids"]
    labels = [-100] * len(p_ids) + ids[len(p_ids):]          # 답변 토큰만 학습
    return {"input_ids": ids, "labels": labels[:len(ids)]}

ds = Dataset.from_list(rows).map(encode, remove_columns=["q", "a"]).shuffle(seed=0)
print(ds, "| 평균 길이", sum(map(len, ds["input_ids"])) / len(ds))

def collate(batch):
    L = max(len(b["input_ids"]) for b in batch)
    pad = lambda seq, v: seq + [v] * (L - len(seq))
    return {"input_ids": torch.tensor([pad(b["input_ids"], tok.pad_token_id) for b in batch]),
            "labels": torch.tensor([pad(b["labels"], -100) for b in batch]),
            "attention_mask": torch.tensor([[1] * len(b["input_ids"]) + [0] * (L - len(b["input_ids"])) for b in batch])}

## 4. 학습 루프 — Paged 8-bit AdamW + 그래디언트 누적 [15][36]
학습 원리가 그대로 보이도록 직접 작성한 루프입니다. (TRL `SFTTrainer` 버전은 아래 부록)

In [ ]:
import bitsandbytes as bnb
from torch.utils.data import DataLoader
opt = bnb.optim.PagedAdamW8bit([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.0)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / 5) * 0.5 * (1 + math.cos(math.pi * min(s, STEPS) / STEPS)))
dl = DataLoader(ds, batch_size=BATCH, shuffle=True, collate_fn=collate)
torch.cuda.reset_peak_memory_stats()
model.train(); it = iter(dl); t0 = time.perf_counter(); ntok = 0
for step in range(STEPS):
    for _ in range(ACCUM):
        try:
            b = next(it)
        except StopIteration:
            it = iter(dl); b = next(it)
        b = {k: v.to(model.device) for k, v in b.items()}
        with torch.autocast("cuda", dtype=torch.bfloat16):
            loss = model(**b).loss / ACCUM
        loss.backward(); ntok += int(b["attention_mask"].sum())
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
    if step % 5 == 0 or step == STEPS - 1:
        el = time.perf_counter() - t0
        print(f"step {step:3d} | loss {loss.item()*ACCUM:.4f} | {ntok/el:6.0f} tok/s | peak {torch.cuda.max_memory_allocated()/2**30:.2f} GB")

In [ ]:
model.eval(); model.config.use_cache = True
for p in EVAL_PROMPTS:
    print("Q:", p)
    print("  학습 전:", before[p][:200].replace("\n", " "))
    print("  학습 후:", chat(p)[:200].replace("\n", " "), "\n")
model.save_pretrained(OUT_DIR); tok.save_pretrained(OUT_DIR)
print("어댑터 저장:", OUT_DIR, f"({sum(os.path.getsize(os.path.join(OUT_DIR, f)) for f in os.listdir(OUT_DIR))/2**20:.0f} MB)")

### 어댑터 활용
- **vLLM 으로 서빙**: 어댑터 폴더를 서빙 컨테이너에 마운트하고 `--enable-lora --lora-modules lab=/adapters/qlora-adapter` 추가 → API 에서 `model="lab"` 으로 호출
- **병합**: BF16 베이스를 다시 로드 → `PeftModel.from_pretrained(base, OUT_DIR).merge_and_unload()` → `save_pretrained` (12GB 에서는 7B BF16 병합을 CPU 에서 수행: `device_map="cpu"`)

In [ ]:
# 메모리 정리 (다음 섹션/다른 노트북 전에 실행)
del model, opt; gc.collect(); torch.cuda.empty_cache()
print(f"VRAM 사용 {torch.cuda.memory_allocated()/2**30:.2f} GB")

## 부록 A. TRL `SFTTrainer` 버전
TRL 은 버전마다 인자 이름이 바뀝니다. 아래는 최근 API 기준이며, 오류가 나면 `trl.__version__` 을 확인하세요.

In [ ]:
RUN_SFT = False      # True 로 바꿔 실행 (위 섹션 1~2 로 model 을 다시 만든 뒤)
if RUN_SFT:
    import trl
    from trl import SFTTrainer, SFTConfig
    print("trl", trl.__version__)
    sft_ds = Dataset.from_list([{"messages": [{"role": "user", "content": q}, {"role": "assistant", "content": a}]} for q, a in facts] * 20)
    cfg = SFTConfig(output_dir="outputs/sft", per_device_train_batch_size=BATCH, gradient_accumulation_steps=ACCUM,
                    learning_rate=LR, max_steps=STEPS, bf16=True, logging_steps=5, optim="paged_adamw_8bit",
                    gradient_checkpointing=True, report_to="none")
    trainer = SFTTrainer(model=model, args=cfg, train_dataset=sft_ds, processing_class=tok)
    trainer.train()

## 부록 B. ZeRO-Offload — 전체 파라미터 학습을 CPU RAM 으로 확장 [18]
LoRA 가 아닌 **전체 파라미터** 학습은 Adam 상태(파라미터당 12바이트)가 VRAM 을 압도합니다.
DeepSpeed ZeRO-2 + Offload 는 옵티마이저 상태와 업데이트 계산을 **CPU RAM(이 서버 30GB) + CPUAdam** 으로 보내
1.5B 모델 전체 학습을 12GB 안에서 가능하게 합니다. (첫 실행 시 CPUAdam 을 JIT 컴파일 — 1~3분)

In [ ]:
ds_cfg = {
    "train_micro_batch_size_per_gpu": 2, "gradient_accumulation_steps": 4,
    "bf16": {"enabled": True},
    "zero_optimization": {"stage": 2, "offload_optimizer": {"device": "cpu", "pin_memory": True},
                          "overlap_comm": True, "contiguous_gradients": True},
    "optimizer": {"type": "AdamW", "params": {"lr": 1e-5, "weight_decay": 0.0}},
    "gradient_clipping": 1.0, "steps_per_print": 5,
}
os.makedirs("outputs", exist_ok=True)
json.dump(ds_cfg, open("outputs/ds_zero2_offload.json", "w"), indent=1)
open("outputs/zero_offload_demo.py", "w").write('''
import time, torch, deepspeed, psutil
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", torch_dtype=torch.bfloat16)
model.gradient_checkpointing_enable()
engine, _, _, _ = deepspeed.initialize(model=model, model_parameters=model.parameters(),
                                       config="outputs/ds_zero2_offload.json")
x = torch.randint(0, 32000, (2, 512), device=engine.device)
t = time.perf_counter()
for step in range(12):
    loss = engine(input_ids=x, labels=x).loss
    engine.backward(loss); engine.step()
    if engine.is_gradient_accumulation_boundary():
        print(f"step {step:2d} loss {loss.item():.3f} | GPU {torch.cuda.max_memory_allocated()/2**30:.2f} GB"
              f" | CPU RAM {psutil.Process().memory_info().rss/2**30:.1f} GB | {time.perf_counter()-t:.0f}s", flush=True)
''')
print("작성 완료 → 다음 셀 실행 (VRAM 비어 있어야 함)")

In [ ]:
!cd {os.getcwd()} && deepspeed --num_gpus 1 outputs/zero_offload_demo.py 2>&1 | grep -E "step|Error|error|OOM" | head -20